# 02 — Training and Controlled Experiments

**Goal:** train the same U-Net under a small, documented experiment matrix.

The default matrix compares augmentation and loss-function choices. The script selects checkpoints using validation loss and evaluates the held-out test set after training.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))
import yaml, torch
from src.data import build_data_bundle
from src.model import UNet, count_parameters
from src.losses import build_loss

cfg = yaml.safe_load(Path("../configs/default.yaml").read_text())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Experiment matrix:")
for exp in cfg["experiments"]:
    print(exp)

## 1. Understand the losses

- Cross-Entropy compares the predicted class distribution with the target class at each pixel.
- Dice loss optimizes overlap and can be useful when class areas are imbalanced.
- Combined loss adds Cross-Entropy and Dice loss with fixed weights.

The combined objective in this project is `0.5 × CE + 0.5 × Dice`. This is an experimental setting, not a universally optimal choice.

In [ ]:
from src.losses import MulticlassDiceLoss, CombinedLoss
logits = torch.randn(2, 3, 32, 32, requires_grad=True)
target = torch.randint(0, 3, (2, 32, 32))
for name, loss_fn in [
    ("Cross-Entropy", torch.nn.CrossEntropyLoss()),
    ("Dice", MulticlassDiceLoss(3)),
    ("Combined", CombinedLoss(3))
]:
    value = loss_fn(logits, target)
    print(name, float(value.detach()))

## 2. Run the experiment matrix

Run this from the repository root in a terminal:

```bash
python -m src.run_experiments --config configs/default.yaml
```

Training may take substantial time on CPU. Start with the conservative defaults. Do not change multiple variables at once if you want interpretable comparisons.

In [ ]:
# Optional: run the experiment runner from a notebook.
# Uncomment only after confirming the environment and available disk space.
# %cd ..
# !python -m src.run_experiments --config configs/default.yaml
# %cd notebooks

## 3. Read the measured result table

The runner writes `results/tables/experiment_comparison.csv`. The next cell reads it only if it exists. If the file is absent, that means the experiments have not yet completed; no scores are fabricated.

In [ ]:
import pandas as pd
results_path = Path("../results/tables/experiment_comparison.csv")
if results_path.exists():
    results = pd.read_csv(results_path)
    display(results)
else:
    print("No measured results yet. Run the experiment matrix from the repository root.")

## 4. Interpretation questions

1. Did augmentation improve validation and test metrics?
2. Did Dice loss improve the boundary class?
3. Did the combined loss improve the macro metrics or only some classes?
4. Is the best validation-loss checkpoint also best on test metrics?
5. Are differences large enough to matter, or could they be seed variation?

Do not write that one method is superior unless your measured evidence supports that conclusion.